# 05 · Avaliação por injeção sintética de anomalias

Avaliação quantitativa do detector: price shocks são injetados em posições conhecidas
da série de teste; Precision, Recall e F1 são calculados contra o ground-truth sintético.
Compara threshold estático vs. dinâmico e os 4 ativos (ADR-0006).

## Setup

In [ ]:
# --- Colab ---
# !git clone https://github.com/Cerne17/NeuraTrade.git
# %cd NeuraTrade
# !pip install -e .

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import CONFIG, set_seeds
from src import data, preprocessing as pp, train as T, detect as D
from src.evaluate import inject_price_shocks, labels_to_window_labels, compute_metrics
from src.viz import plot_error_distribution, plot_error_timeseries, save_fig

set_seeds()
TICKERS = CONFIG['tickers']
W = CONFIG['preprocessing']['window_size']

pre    = {t: pp.preprocess_ticker(df) for t, df in data.load_all().items()}
models = {t: T.load_model(t) for t in TICKERS}
print('modelos carregados:', list(models))

## Erros de reconstrução — referência (sem injeção)

Linha de base: erros normais do modelo treinado. O threshold estático é calculado aqui.

In [ ]:
err_tr = {t: D.reconstruction_error(models[t], pre[t]['X_train']) for t in TICKERS}
err_te = {t: D.reconstruction_error(models[t], pre[t]['X_test'])  for t in TICKERS}
thr_s  = {t: D.static_threshold(err_tr[t]) for t in TICKERS}
thr_d  = {t: D.dynamic_threshold(err_te[t]) for t in TICKERS}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, t in zip(axes.ravel(), TICKERS):
    plot_error_distribution(err_tr[t], err_te[t], thr_s[t], t, ax=ax)
fig.suptitle('Distribuição do erro de reconstrução (MAE) — treino vs. teste')
fig.tight_layout()
save_fig(fig, 'eval_distribuicao_erro')
plt.show()

## Injeção de price shocks (issue #22)

50 choques de magnitude `price_shock=0.15` são inseridos em posições aleatórias da série
de teste escalada. O modelo é então aplicado sobre a série perturbada.

In [ ]:
# reconstroi serie 1D de teste escalada por ativo
def test_series_1d(t):
    df = data.load_ticker(t)
    r = pp.log_returns(df)
    r_train, r_test = pp.temporal_split(r)
    scaler = pp.fit_scaler(r_train)
    return pp.apply_scaler(scaler, r_test), pre[t]['scaler']

injected = {}
for t in TICKERS:
    ts, _ = test_series_1d(t)
    perturbed, labels, positions = inject_price_shocks(ts)
    X_pert = pp.make_windows(perturbed)
    err_pert = D.reconstruction_error(models[t], X_pert)
    injected[t] = dict(labels=labels, positions=positions, err_pert=err_pert)
    print(f'{t}: {len(positions)} choques | err medio perturbado={err_pert.mean():.4f} vs normal={err_te[t].mean():.4f}')

## Precision, Recall e F1 por ativo (issue #23)

Threshold estático e dinâmico comparados. O dinâmico é calculado sobre os erros da série
perturbada (o detector não vê os labels — só os erros).

In [ ]:
rows = []
for t in TICKERS:
    ep  = injected[t]['err_pert']
    lbl = injected[t]['labels']
    wl  = labels_to_window_labels(lbl)

    thr_d_pert = D.dynamic_threshold(ep)

    for thr_name, thr_val in [('estatico', thr_s[t]), ('dinamico', thr_d_pert)]:
        flags = D.flag_anomalies(ep, thr_val)
        m = compute_metrics(flags, wl)
        rows.append({'ticker': t, 'threshold': thr_name, **{k: round(v, 3) for k, v in m.items()}})

df_metrics = pd.DataFrame(rows).set_index(['ticker', 'threshold'])
df_metrics

## Evolução temporal do erro com posições injetadas (issue #24)

Série do erro com os choques injetados marcados em laranja — permite verificar visualmente
se os picos de erro coincidem com os pontos injetados.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for ax, t in zip(axes.ravel(), TICKERS):
    ep   = injected[t]['err_pert']
    wl   = labels_to_window_labels(injected[t]['labels'])
    fl   = D.flag_anomalies(ep, thr_s[t])
    d    = pre[t]['test_index'][W-1 : W-1+len(ep)]

    plot_error_timeseries(ep, d, thr_s[t], fl, t, ax=ax)

    # marca as janelas com choque injetado em laranja
    inj_mask = wl.astype(bool)
    ax.scatter(d[inj_mask], ep[inj_mask], color='orange', s=8, zorder=4, label='injetado')
    ax.legend(fontsize=7)

fig.suptitle('Erro de reconstrução — série perturbada (laranja=choque injetado, vermelho=detectado)')
fig.tight_layout()
save_fig(fig, 'eval_serie_perturbada')
plt.show()

## Conclusões

- O threshold **estático** tende a ter precision maior (limiar fixo não se adapta a ruído local);
  o **dinâmico** ajusta-se ao regime, podendo variar recall por ativo.
- AMER3 (varejo) apresenta erros de base mais elevados após a fraude — o choque sintético
  compete com anomalias reais, reduzindo precision.
- Magnitudes provisórias (`price_shock=0.15`): calibrar em função do desvio-padrão do
  retorno real de cada ativo (ADR-0006).
- Próximo (M6): correlação das anomalias com eventos históricos e comparação setorial.